# 33 — Baseline Correctness & Fairness Verification

Verifies that each re-implemented baseline is correct and evaluated fairly.

| Section | Method | Check |
|---------|--------|-------|
| 1 | ICWS (nb26) | Sketch collision rate ≈ exact WJ; cross-check vs datasketch |
| 2 | AE recon (nb27) + TripletAE (nb28) | WJ(z_a,z_p) vs WJ(x_a,x_p) Spearman correlation |
| 3 | Deep Binary Hash (nb29) | Bit similarity vs exact WJ Spearman; bit balance check |
| 4 | Matryoshka MLP (nb30) | Recall degrades gracefully: 128 < 256 < 512 dims |
| 5 | All methods | Fairness audit: same dataset, GT, eval function |

**Pass criteria printed at end of each section.**

In [1]:
import sys, pickle, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.stats import spearmanr

sys.path.append('/raid/ruban/hpmlproj/term_project/SigSpatial')
from sota_experiment_common import load_dataset, l1_simplex, QUERY_START_10K

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'device={device}')

qt, gt, query_start, corpus_qt, query_qt, corpus_sums = load_dataset('10k')
qt_norm = l1_simplex(qt.copy())

N_PAIRS = 500
rng = np.random.default_rng(42)

# Sample 500 random corpus pairs
pair_a_ids = rng.integers(0, query_start, N_PAIRS)
pair_b_ids = rng.integers(0, query_start, N_PAIRS)

xa = qt_norm[pair_a_ids]
xb = qt_norm[pair_b_ids]

def exact_wj_numpy(a, b):
    mins = np.minimum(a, b).sum(axis=1)
    maxs = np.maximum(a, b).sum(axis=1)
    return mins / np.maximum(maxs, 1e-10)

exact_wj = exact_wj_numpy(xa, xb)
print(f'exact WJ: mean={exact_wj.mean():.4f}  min={exact_wj.min():.4f}  max={exact_wj.max():.4f}')

device=cuda:0
dataset=10k | qt=(10000, 18499) | corpus=(8000, 18499) | queries=(2000, 18499)
exact WJ: mean=0.1266  min=0.0000  max=0.8097


---
## Section 1 — ICWS (nb26) correctness

Our ICWS implementation computes sketch collision rate ≈ WJ.  
Cross-check: compare Spearman rank correlation of our sketches vs datasketch WeightedMinHash vs exact WJ.

**Pass**: Spearman(our_icws, exact_wj) > 0.90

In [2]:
# ── Our ICWS implementation (copied verbatim from nb26) ──────────────────────
def icws_signatures(x, num_samples=512, seed=42, batch_size=128):
    rng_local = np.random.default_rng(seed)
    n_dim = x.shape[1]
    U  = rng_local.uniform(0, 1, (num_samples, n_dim)).astype(np.float32)
    V  = rng_local.uniform(0, 1, (num_samples, n_dim)).astype(np.float32)
    Y  = rng_local.uniform(0, 1, (num_samples, n_dim)).astype(np.float32)
    
    sig_idx = np.empty((len(x), num_samples), dtype=np.int32)
    sig_t   = np.empty((len(x), num_samples), dtype=np.float32)

    for start in range(0, len(x), batch_size):
        xb_chunk = x[start:start + batch_size]  # (B, D)
        # ICWS: for each sample k and dim j, compute t = floor(log(x_j)/log(u_kj) + 1)
        # collision if argmin_t is same for two vectors → approx WJ
        log_x = np.log(np.maximum(xb_chunk, 1e-10))[:, np.newaxis, :]   # (B,1,D)
        log_u = np.log(np.maximum(U, 1e-10))[np.newaxis, :, :]           # (1,K,D)
        t_k = np.floor(log_x / log_u + 1e-10).astype(np.float32)        # (B,K,D)
        log_v = np.log(np.maximum(V, 1e-10))[np.newaxis, :, :]           # (1,K,D)
        log_y = np.log(np.maximum(Y, 1e-10))[np.newaxis, :, :]           # (1,K,D)
        a_k = t_k - log_v / log_u
        idx_min = np.argmin(a_k, axis=2)   # (B, K)
        t_min   = t_k[np.arange(len(xb_chunk))[:, None],
                      np.arange(num_samples)[None, :],
                      idx_min]              # (B, K)
        sig_idx[start:start + len(xb_chunk)] = idx_min
        sig_t  [start:start + len(xb_chunk)] = t_min
    return sig_idx, sig_t

def icws_similarity(sig_a, sig_b):
    idx_a, t_a = sig_a
    idx_b, t_b = sig_b
    matches = (idx_a == idx_b) & (t_a == t_b)
    return matches.mean(axis=1).astype(np.float32)

print('Computing ICWS signatures for 500-pair subset...')
xy = np.vstack([xa, xb])  # (1000, D)
sig_idx, sig_t = icws_signatures(xy, num_samples=512, seed=42)

sig_a = (sig_idx[:N_PAIRS], sig_t[:N_PAIRS])
sig_b = (sig_idx[N_PAIRS:], sig_t[N_PAIRS:])
our_icws_sim = icws_similarity(sig_a, sig_b)

rho_ours, p_ours = spearmanr(our_icws_sim, exact_wj)
print(f'Spearman(our_icws, exact_wj) = {rho_ours:.4f}  (p={p_ours:.2e})')

Computing ICWS signatures for 500-pair subset...
Spearman(our_icws, exact_wj) = 0.9695  (p=2.05e-306)


In [3]:
# ── datasketch cross-check ───────────────────────────────────────────────────
try:
    from datasketch import WeightedMinHash, WeightedMinHashGenerator
    DATASKETCH_AVAILABLE = True
    print('datasketch available — running cross-check')
except ImportError:
    DATASKETCH_AVAILABLE = False
    print('datasketch not installed — skipping cross-check (install: pip install datasketch)')

if DATASKETCH_AVAILABLE:
    wmg = WeightedMinHashGenerator(xa.shape[1], sample_size=512, seed=42)
    datasketch_sims = []
    for i in range(N_PAIRS):
        h1 = wmg.minhash(xa[i].astype(float))
        h2 = wmg.minhash(xb[i].astype(float))
        datasketch_sims.append(h1.jaccard(h2))
    datasketch_sims = np.array(datasketch_sims)
    rho_ds, p_ds = spearmanr(datasketch_sims, exact_wj)
    rho_cross, _ = spearmanr(our_icws_sim, datasketch_sims)
    print(f'Spearman(datasketch, exact_wj)  = {rho_ds:.4f}  (p={p_ds:.2e})')
    print(f'Spearman(our_icws, datasketch)  = {rho_cross:.4f}')

print()
PASS = rho_ours > 0.90
print(f'[Section 1] ICWS correctness: {"PASS" if PASS else "FAIL"}  '
      f'(Spearman={rho_ours:.4f}, threshold=0.90)')

datasketch not installed — skipping cross-check (install: pip install datasketch)

[Section 1] ICWS correctness: PASS  (Spearman=0.9695, threshold=0.90)


---
## Section 2 — AE reconstruction (nb27) + TripletAE (nb28)

Load each checkpoint, embed the 500-pair subset, compute WJ(z_a, z_p) in embedding space,  
compare Spearman rank correlation vs exact WJ(x_a, x_p).

**Pass**: Spearman > 0.80 for AE recon, > 0.85 for TripletAE

In [4]:
IN_DIM = qt_norm.shape[1]
OUT_DIM = 512

# ── WJAutoencoder (nb27) ─────────────────────────────────────────────────────
class WJAutoencoder(nn.Module):
    def __init__(self, in_dim, out_dim=512):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, 4096, bias=False), nn.BatchNorm1d(4096), nn.ReLU(),
            nn.Linear(4096, 1024, bias=False), nn.BatchNorm1d(1024), nn.ReLU(),
            nn.Linear(1024, out_dim, bias=False), nn.BatchNorm1d(out_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(out_dim, 1024), nn.ReLU(),
            nn.Linear(1024, 4096), nn.ReLU(),
            nn.Linear(4096, in_dim), nn.ReLU(),
        )
    def encode(self, x):
        z = F.relu(self.encoder(x))
        return z / z.sum(dim=1, keepdim=True).clamp(min=1e-10)
    def forward(self, x):
        z = self.encode(x)
        rec = self.decoder(z)
        rec = rec / rec.sum(dim=1, keepdim=True).clamp(min=1e-10)
        return z, rec

# ── TripletAE (nb28) ─────────────────────────────────────────────────────────
class TripletAE(nn.Module):
    def __init__(self, in_dim, out_dim=512):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, 4096, bias=False), nn.BatchNorm1d(4096), nn.ReLU(),
            nn.Linear(4096, 1024, bias=False), nn.BatchNorm1d(1024), nn.ReLU(),
            nn.Linear(1024, out_dim, bias=False), nn.BatchNorm1d(out_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(out_dim, 1024, bias=False), nn.BatchNorm1d(1024), nn.ReLU(),
            nn.Linear(1024, in_dim, bias=False),
        )
    def encode(self, x):
        z = F.relu(self.encoder(x))
        return z / z.sum(dim=1, keepdim=True).clamp(min=1e-10)
    def forward(self, x):
        z = self.encode(x)
        return z, F.relu(self.decoder(z))

def embed_pairs(model, xa, xb):
    model.eval()
    with torch.no_grad():
        ta = torch.tensor(xa, dtype=torch.float32, device=device)
        tb = torch.tensor(xb, dtype=torch.float32, device=device)
        za = model.encode(ta).cpu().numpy()
        zb = model.encode(tb).cpu().numpy()
    return za, zb

def wj_numpy_pairs(za, zb):
    mins = np.minimum(za, zb).sum(axis=1)
    maxs = np.maximum(za, zb).sum(axis=1)
    return mins / np.maximum(maxs, 1e-10)

results_s2 = {}

for name, ModelClass, ckpt in [
    ('AE_recon (nb27)',  WJAutoencoder, '/tmp/best_sota_autoencoder_wj_512.pt'),
    ('TripletAE (nb28)', TripletAE,     '/tmp/best_sota_triplet_autoencoder_wj_512.pt'),
]:
    import os
    if not os.path.exists(ckpt):
        print(f'[{name}] checkpoint missing: {ckpt}')
        continue
    model = ModelClass(IN_DIM, OUT_DIM).to(device)
    model.load_state_dict(torch.load(ckpt, map_location=device))
    za, zb = embed_pairs(model, xa, xb)
    emb_wj = wj_numpy_pairs(za, zb)
    rho, p = spearmanr(emb_wj, exact_wj)
    results_s2[name] = rho
    print(f'[{name}]  WJ emb mean={emb_wj.mean():.4f}  '
          f'Spearman(emb_wj, exact_wj)={rho:.4f}  (p={p:.2e})')
    del model
    torch.cuda.empty_cache()

print()
for name, rho in results_s2.items():
    threshold = 0.85 if 'Triplet' in name else 0.80
    PASS = rho > threshold
    print(f'[Section 2] {name}: {"PASS" if PASS else "FAIL"}  '
          f'(Spearman={rho:.4f}, threshold={threshold})')

/tmp/ipykernel_2666915/1781950450.py:72: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(ckpt, map_location=device))


[AE_recon (nb27)]  WJ emb mean=0.1899  Spearman(emb_wj, exact_wj)=0.7537  (p=8.03e-93)
[TripletAE (nb28)]  WJ emb mean=0.0746  Spearman(emb_wj, exact_wj)=0.1153  (p=9.85e-03)

[Section 2] AE_recon (nb27): FAIL  (Spearman=0.7537, threshold=0.8)
[Section 2] TripletAE (nb28): FAIL  (Spearman=0.1153, threshold=0.85)


---
## Section 3 — Deep Binary Hashing (nb29)

Two checks:
1. **Bit similarity vs exact WJ**: `bit_sim(z_a, z_b) = 1 - |z_a - z_b|.mean()` should rank-correlate with WJ (Spearman > 0.75)
2. **Bit balance**: mean activation per bit across 500 samples should be 0.3–0.7 (not collapsed)

**Pass**: both criteria met

In [5]:
import os

class BinaryHashMLP(nn.Module):
    def __init__(self, in_dim, bits=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 4096, bias=False), nn.BatchNorm1d(4096), nn.ReLU(),
            nn.Linear(4096, 1024, bias=False), nn.BatchNorm1d(1024), nn.ReLU(),
            nn.Linear(1024, bits),
        )
    def encode_prob(self, x):
        return torch.sigmoid(self.net(x))
    def encode(self, x):
        return (self.encode_prob(x) > 0.5).float()

ckpt = '/tmp/best_sota_deep_binary_hash_wj_512.pt'
if not os.path.exists(ckpt):
    print(f'checkpoint missing: {ckpt}')
else:
    model = BinaryHashMLP(IN_DIM, OUT_DIM).to(device)
    model.load_state_dict(torch.load(ckpt, map_location=device))
    model.eval()

    with torch.no_grad():
        ta = torch.tensor(xa, dtype=torch.float32, device=device)
        tb = torch.tensor(xb, dtype=torch.float32, device=device)
        # Use soft probs for rank correlation (harder signal than hard bits)
        pa = model.encode_prob(ta).cpu().numpy()
        pb = model.encode_prob(tb).cpu().numpy()
        za = model.encode(ta).cpu().numpy()
        zb = model.encode(tb).cpu().numpy()

    # Check 1: rank correlation
    bit_sim_hard  = 1.0 - np.abs(za - zb).mean(axis=1)
    bit_sim_soft  = 1.0 - np.abs(pa - pb).mean(axis=1)
    rho_hard, _ = spearmanr(bit_sim_hard, exact_wj)
    rho_soft, _ = spearmanr(bit_sim_soft, exact_wj)
    print(f'Spearman(bit_sim_hard, exact_wj) = {rho_hard:.4f}')
    print(f'Spearman(bit_sim_soft, exact_wj) = {rho_soft:.4f}')

    # Check 2: bit balance on corpus sample (500 vectors)
    corpus_sample = qt_norm[rng.integers(0, query_start, 500)]
    with torch.no_grad():
        ts = torch.tensor(corpus_sample, dtype=torch.float32, device=device)
        zs = model.encode(ts).cpu().numpy()
    bit_means = zs.mean(axis=0)
    frac_balanced = ((bit_means > 0.3) & (bit_means < 0.7)).mean()
    print(f'Bit balance: {frac_balanced*100:.1f}% of bits in [0.3, 0.7]  '
          f'(mean={bit_means.mean():.3f}, min={bit_means.min():.3f}, max={bit_means.max():.3f})')

    del model
    torch.cuda.empty_cache()

    print()
    PASS1 = rho_soft > 0.75
    PASS2 = frac_balanced > 0.50
    print(f'[Section 3a] Rank correlation: {"PASS" if PASS1 else "FAIL"}  (Spearman_soft={rho_soft:.4f}, threshold=0.75)')
    print(f'[Section 3b] Bit balance:      {"PASS" if PASS2 else "FAIL"}  ({frac_balanced*100:.1f}% balanced, threshold=50%)')

/tmp/ipykernel_2666915/1404523071.py:21: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(ckpt, map_location=device))


Spearman(bit_sim_hard, exact_wj) = 0.1537
Spearman(bit_sim_soft, exact_wj) = 0.1548
Bit balance: 100.0% of bits in [0.3, 0.7]  (mean=0.499, min=0.356, max=0.636)

[Section 3a] Rank correlation: FAIL  (Spearman_soft=0.1548, threshold=0.75)
[Section 3b] Bit balance:      PASS  (100.0% balanced, threshold=50%)


---
## Section 4 — Matryoshka MLP (nb30): graceful degradation

Recall@10 at dims [128, 256, 512] should be monotonically increasing.
If 128-dim ≥ 512-dim, the prefix-norm training is broken.

**Pass**: R@10(128) < R@10(256) < R@10(512)

In [6]:
from sota_experiment_common import nmslib_neighbors, eval_recall

class MatryoshkaWJMLP(nn.Module):
    def __init__(self, in_dim, out_dim=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 4096, bias=False), nn.BatchNorm1d(4096), nn.ReLU(),
            nn.Linear(4096, 1024, bias=False), nn.BatchNorm1d(1024), nn.ReLU(),
            nn.Linear(1024, out_dim, bias=False), nn.BatchNorm1d(out_dim),
        )
    def encode(self, x):
        z = F.relu(self.net(x))
        return z / z.sum(dim=1, keepdim=True).clamp(min=1e-10)
    def forward(self, x):
        return self.encode(x)

def prefix_norm(z, d):
    p = z[:, :d]
    return p / p.sum(axis=1, keepdims=True).clip(min=1e-10)

ckpt = '/tmp/best_sota_matryoshka_mlp_wj_512.pt'
results_s4 = {}

if not os.path.exists(ckpt):
    print(f'checkpoint missing: {ckpt}')
else:
    model = MatryoshkaWJMLP(IN_DIM, OUT_DIM).to(device)
    model.load_state_dict(torch.load(ckpt, map_location=device))
    model.eval()

    # Full 512-dim embeddings
    with torch.no_grad():
        all_embs = []
        for start in range(0, len(qt_norm), 512):
            x = torch.tensor(qt_norm[start:start+512], dtype=torch.float32, device=device)
            all_embs.append(model.encode(x).cpu().numpy())
    all_embs = np.vstack(all_embs).astype(np.float32)

    for d in [128, 256, 512]:
        embs_d = prefix_norm(all_embs, d)
        corpus_d = embs_d[:query_start]
        query_d  = embs_d[query_start:]
        nbrs, _ = nmslib_neighbors(corpus_d, query_d, k=500, threads=16)
        metrics = eval_recall(gt, nbrs, query_start, max_k=500)
        r10 = metrics[10]
        results_s4[d] = r10
        print(f'dim={d:4d}  R@10={r10:.4f}')

    del model
    torch.cuda.empty_cache()

    print()
    if len(results_s4) == 3:
        PASS = results_s4[128] < results_s4[256] < results_s4[512]
        print(f'[Section 4] Matryoshka graceful degradation: {"PASS" if PASS else "FAIL"}')
        if not PASS:
            print('  WARNING: recall does not increase with dim — prefix-norm may be incorrect')

/tmp/ipykernel_2666915/1931734943.py:28: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(ckpt, map_location=device))

0%   10   20   30   40  

dim= 128  R@10=0.4523



0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
*

dim= 256  R@10=0.4520



0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
*

dim= 512  R@10=0.4548

[Section 4] Matryoshka graceful degradation: FAIL


---
## Section 5 — Fairness audit: identical evaluation across all methods

Checks:
- All pickles use dataset_name='10k'
- All have same number of queries
- All rerank rows use same candidate_k values (500 or 1000)
- All were evaluated with the same `eval_recall` from sota_experiment_common

In [7]:
from pathlib import Path

BACKUP_DIR = Path('/raid/ruban/hpmlproj/term_project/results_10k')

result_files = {
    'pca':                BACKUP_DIR / 'results_sota_pca_wj_512.pkl',
    'random_proj':        BACKUP_DIR / 'results_sota_random_proj_wj_512.pkl',
    'nmf':                BACKUP_DIR / 'results_sota_nmf_wj_512.pkl',
    'icws':               BACKUP_DIR / 'results_sota_icws_512.pkl',
    'autoencoder':        BACKUP_DIR / 'results_sota_autoencoder_wj_512.pkl',
    'triplet_autoencoder':BACKUP_DIR / 'results_sota_triplet_autoencoder_wj_512.pkl',
    'deep_binary_hash':   BACKUP_DIR / 'results_sota_deep_binary_hash_wj_512.pkl',
    'matryoshka':         BACKUP_DIR / 'results_sota_matryoshka_mlp_wj_512.pkl',
    'mlp_wj_native_no_fn':BACKUP_DIR / 'results_mlp_wj_native_no_fn_filter.pkl',
    'mlp_wj_reg_maxpos100':BACKUP_DIR / 'results_mlp_wj_reg_maxpos100.pkl',
}

print(f'Expected query count: {len(query_qt)} queries')
print()

all_pass = True
for name, path in result_files.items():
    if not path.exists():
        print(f'  [MISSING]  {name}: {path}')
        all_pass = False
        continue
    with open(path, 'rb') as f:
        payload = pickle.load(f)
    # Must have '10k' key
    if '10k' not in payload:
        print(f'  [FAIL] {name}: no "10k" key in pickle')
        all_pass = False
        continue
    runs = payload['10k']
    run_names = list(runs.keys())
    # Check candidate_k values in rerank runs
    candidate_ks = set()
    for rn, metrics in runs.items():
        ck = metrics.get('candidate_k')
        if ck: candidate_ks.add(ck)
    ck_str = str(sorted(candidate_ks)) if candidate_ks else 'no-rerank'
    meta = payload.get('_meta', {}).get(run_names[0], {})
    ts = meta.get('time', 'unknown')
    r10_vals = [metrics.get(10, float('nan')) for metrics in runs.values()]
    r10_str = '  '.join(f'{v:.4f}' for v in r10_vals if not np.isnan(v))
    print(f'  [OK] {name:30s}  runs={len(runs)}  candidate_ks={ck_str:20s}  R@10=[{r10_str}]  time={ts}')

print()
print(f'[Section 5] Fairness audit: {"ALL PASS" if all_pass else "ISSUES FOUND — see above"}')

Expected query count: 2000 queries

  [OK] pca                             runs=3  candidate_ks=[500, 1000]           R@10=[0.4321  0.9608  0.9694]  time=2026-06-08 22:30:06
  [OK] random_proj                     runs=3  candidate_ks=[500, 1000]           R@10=[0.6869  0.9964  0.9966]  time=2026-06-09 07:03:18
  [OK] nmf                             runs=3  candidate_ks=[500, 1000]           R@10=[0.6382  0.9964  0.9966]  time=2026-06-08 22:39:18
  [OK] icws                            runs=1  candidate_ks=no-rerank             R@10=[0.8402]  time=2026-06-08 22:58:53
  [OK] autoencoder                     runs=3  candidate_ks=[500, 1000]           R@10=[0.6653  0.9961  0.9966]  time=2026-06-09 04:57:01
  [OK] triplet_autoencoder             runs=3  candidate_ks=[500, 1000]           R@10=[0.7114  0.9966  0.9966]  time=2026-06-09 07:26:04
  [OK] deep_binary_hash                runs=1  candidate_ks=no-rerank             R@10=[0.4322]  time=2026-06-09 05:19:05
  [OK] matryoshka             

---
## Summary

In [8]:
print('=' * 60)
print('VERIFICATION SUMMARY')
print('=' * 60)
print('Run each section above and check the PASS/FAIL lines.')
print()
print('Section 1  ICWS             Spearman(sketch, exact_wj) > 0.90')
print('Section 2a AE recon (nb27)  Spearman(emb_wj, exact_wj) > 0.80')
print('Section 2b TripletAE (nb28) Spearman(emb_wj, exact_wj) > 0.85')
print('Section 3a Deep Binary Hash Spearman(bit_sim, exact_wj) > 0.75')
print('Section 3b Deep Binary Hash >50% of bits balanced [0.3, 0.7]')
print('Section 4  Matryoshka       R@10(128) < R@10(256) < R@10(512)')
print('Section 5  All methods      Same dataset/GT/eval — all present')

VERIFICATION SUMMARY
Run each section above and check the PASS/FAIL lines.

Section 1  ICWS             Spearman(sketch, exact_wj) > 0.90
Section 2a AE recon (nb27)  Spearman(emb_wj, exact_wj) > 0.80
Section 2b TripletAE (nb28) Spearman(emb_wj, exact_wj) > 0.85
Section 3a Deep Binary Hash Spearman(bit_sim, exact_wj) > 0.75
Section 3b Deep Binary Hash >50% of bits balanced [0.3, 0.7]
Section 4  Matryoshka       R@10(128) < R@10(256) < R@10(512)
Section 5  All methods      Same dataset/GT/eval — all present
